In [ ]:
from langgraph.graph import StateGraph , START , END
from typing  import TypedDict , Annotated , List
import operator 
from langchain_core.messages import BaseMessage , HumanMessage
from langgraph.checkpoint.memory import MemorySaver
from langchain_groq import ChatGroq

In [3]:
from langgraph.graph.message import add_messages

class ChatState(TypedDict):
    messages : Annotated[list[BaseMessage] , add_messages]

In [ ]:
llm = ChatGroq()

def chat_node(state:ChatState):
    messages = state['messages']
    response = llm.invoke(messages)
    return {'messages': [response]}

In [ ]:
checkpointer = MemorySaver()

graph = StateGraph(ChatState)


graph.add_node('chat_node', chat_node)
graph.add_edge(START , 'chat_node')
graph.add_edge('chat_node' , END)

workflow = graph.compile(checkpointer=checkpointer)

In [ ]:
initial_state = [
    HumanMessage(content = "")
]

In [ ]:

final_state = workflow.invoke(initial_state)

In [ ]:
thread_id = 1


while True:
    user_message = input("Type Here")
    
    if user_message.strip().lower() in ['exit' , 'quit' , 'bye']:
        break
    config = {'configurable' :{'thread_id' : thread_id}}
    response = workflow.invoke({'messages':[HumanMessage(content = user_message)]} , config = config)
    print('AI', response['messages'][-1].content)